# 1. Khởi tạo Spark Session

Khởi tạo một SparkSession local tận dụng tối đa số luồng xử lý (local[*]) để phục vụ cho các tác vụ xử lý dữ liệu lớn.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("MovieRecommendation-ETL-Notebook") \
    .master("local[*]") \
    .getOrCreate()
    
print("=> Khởi tạo Spark thành công!")


=> Khởi tạo Spark thành công!


# 2. Nạp và làm sạch dữ liệu Rating (u.data)

Định nghĩa schema tường minh cho tập dữ liệu đánh giá, tiến hành xóa bỏ các bản ghi rỗng (null) và loại bỏ trùng lặp.

In [3]:
rating_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Đường dẫn lùi ra 2 thư mục để vào data
rating_path = "../../data/ml-100k/u.data"
ratings_df = spark.read.csv(rating_path, sep="\t", schema=rating_schema)

# Xóa dữ liệu rỗng và trùng lặp
ratings_df = ratings_df.dropna().dropDuplicates()

print(f"Tổng số lượt đánh giá sau làm sạch: {ratings_df.count()}")
ratings_df.show(5) # In thử 5 dòng ra xem


Tổng số lượt đánh giá sau làm sạch: 100000
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|    292|     515|     4|881103977|
|    178|     248|     4|882823954|
|     32|     294|     3|883709863|
|     57|     419|     3|883698454|
|    229|     328|     1|891632142|
+-------+--------+------+---------+
only showing top 5 rows




## Tối ưu hóa quy trình nạp dữ liệu (ETL)

Thay vì để Spark tự động dò tìm kiểu dữ liệu, chúng ta thực hiện các kỹ thuật tối ưu sau:

* **Định nghĩa Schema sẵn (`schema=rating_schema`):** Giúp Spark nạp dữ liệu trong 1 lần quét duy nhất, thay vì 2 lần (tiết kiệm đáng kể thời gian xử lý file lớn).


* **Xử lý phân tách (`sep="\t"`):** Chỉ định rõ ký tự phân tách là Tab, giúp cấu trúc dữ liệu luôn thẳng hàng và chính xác ngay từ đầu.


* **Chuẩn hóa kiểu dữ liệu số (`IntegerType`):** Ép kiểu dữ liệu về số nguyên ngay khi nạp, đảm bảo dữ liệu sẵn sàng cho các phép tính toán ma trận của thuật toán Machine Learning (ALS) mà không cần chuyển đổi trung gian.

# 3. Nạp và làm sạch dữ liệu Phim (u.item)

Đọc tập thông tin phim với bảng mã hóa ISO-8859-1 nhằm tránh lỗi font. Chỉ trích xuất và chuẩn hóa kiểu dữ liệu cho 2 trường thông tin cốt lõi là mã phim và tên phim.

In [4]:
item_path = "../../data/ml-100k/u.item"

# Đã sửa lỗi Encoding thành ISO-8859-1 chuẩn của Spark 3.5
items_df = spark.read.csv(item_path, sep="|", encoding="ISO-8859-1")

# Chỉ trích xuất 2 cột quan trọng nhất: Mã phim và Tên phim
items_df = items_df.select(
    col("_c0").cast(IntegerType()).alias("movie_id"),
    col("_c1").cast(StringType()).alias("movie_title")
)

items_df = items_df.dropna().dropDuplicates(["movie_id"])

print(f"Tổng số phim sau làm sạch: {items_df.count()}")
items_df.show(5, truncate=False)


Tổng số phim sau làm sạch: 1682
+--------+-----------------+
|movie_id|movie_title      |
+--------+-----------------+
|1       |Toy Story (1995) |
|2       |GoldenEye (1995) |
|3       |Four Rooms (1995)|
|4       |Get Shorty (1995)|
|5       |Copycat (1995)   |
+--------+-----------------+
only showing top 5 rows



## Tối ưu hóa và chuẩn hóa dữ liệu Phim (u.item)

Bước này thực hiện xử lý dữ liệu thô với các kỹ thuật tối ưu cốt lõi sau:

* **Sửa lỗi Font (`encoding="ISO-8859-1"`):** Ngăn chặn tình trạng lỗi hiển thị ký tự đặc biệt đối với các tựa phim nước ngoài, đảm bảo dữ liệu văn bản đọc vào chính xác.
* **Giảm tải bộ nhớ (`.select()`):** Chủ động trích xuất duy nhất 2 cột cần thiết (`movie_id`, `movie_title`) thay vì nạp toàn bộ 24 cột thô, giúp giảm dung lượng RAM khi xử lý phân tán.
* **Loại bỏ trùng lặp tuyệt đối (`.dropDuplicates(["movie_id"])`):** Đảm bảo mã phim (`movie_id`) là duy nhất, đóng vai trò là khóa chính (Primary Key) chuẩn để phục vụ cho các phép nối bảng (`JOIN`) ở các bước sau.

# 4. Xuất ra định dạng Parquet siêu tốc

Lưu trữ kết quả đã làm sạch xuống đĩa dưới định dạng nén cột Parquet. Định dạng này giúp tối ưu hóa hiệu năng đọc/ghi cho các công đoạn tiếp theo như EDA và Train Model.

In [5]:
output_rating_path = "../../data/processed_ratings.parquet"
output_item_path = "../../data/processed_items.parquet"

# Ghi đè file Parquet
ratings_df.write.mode("overwrite").parquet(output_rating_path)
items_df.write.mode("overwrite").parquet(output_item_path)

print("=> Tuyệt vời! Đã xuất thành công 2 file Parquet để dành cho EDA và Machine Learning.")


=> Tuyệt vời! Đã xuất thành công 2 file Parquet để dành cho EDA và Machine Learning.


## Xuất dữ liệu tối ưu ra định dạng Parquet

Bước cuối cùng trong quy trình ETL giúp đóng gói và lưu trữ dữ liệu hiệu năng cao với các ưu điểm:

* **Bảo toàn cấu trúc (`Schema-preserving`):** Parquet tự động lưu kèm kiểu dữ liệu (Số, Chữ) đã cấu hình. Khi các file Notebook sau (EDA, Train Model) đọc lại sẽ nhận diện được ngay mà không cần quét lại dữ liệu.
* **Nén cột tối ưu (`Columnar Storage`):** Dữ liệu được nén chặt giúp tiết kiệm dung lượng đĩa và tăng tốc độ đọc/ghi dữ liệu lên gấp nhiều lần cho hệ thống Big Data.
* **Cơ chế ghi đè an toàn (`mode("overwrite")`):** Đảm bảo quy trình ETL có thể chạy lại nhiều lần một cách độc lập mà không sợ bị trùng lặp hay nhân bản dữ liệu cũ.